### Final Project: Detection of Rare Fetal Congenital Heart Diseases (CHD) Using CARDIUM Ultrasound Images and FetalCLIP: A Deep-Learning Model for Rural Coloradans
#### CSCI 5922
#### Spring 2026
#### Meghna Nag and Vanessa Thorsten
#### Program to Create CLIP Embeddings

#### CARDIUM
The publicly available CARDIUM data and sample code are provided on their GitHub repository (https://github.com/BCV-Uniandes/Cardium#). The original CARDIUM workflow uses a Vision Transformer (ViT) based approach for image processing. In this notebook, we are applying FetalCLIP to the images and saving the outputted embeddings.

Although FetalCLIP also uses a Vision Transformer backbone, it was pretrained using paired fetal ultrasound images and clinical text, allowing it to provide general-purpose ultrasound image representations that can be reused across downstream tasks.

#### FetalCLIP
FetalCLIP provides publicly-available Python code for using the FetalCLIP codebase and pretained model weights on their GitHub repository (https://github.com/BioMedIA-MBZUAI/FetalCLIP). The model-loading and image-embedding approach in notebook was adapted from their example code. We updated this code to process CARDIUM images which are stored in three data folds. The extracted embeddings will be saved and used in our neural network experiments.

In [ ]:
!pip install open-clip-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
# =============================================================================
# Import Packages
# =============================================================================
import os
from pathlib import Path
import json
import torch
from torch.utils.data import Dataset, DataLoader
import open_clip
from PIL import Image
import pandas as pd
import numpy as np

In [ ]:
# =============================================================================
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/CARDIUM"
os.listdir(path)

['CARDIUM dataset-20260318T201738Z-3-001.zip',
 'CARDIUM_dataset-002.tar.gz',
 'cardium_images-003.tar.gz',
 'README_CARDIUM.md',
 'cardium_clinical_data_wnm_translated_final_cleaned.json',
 'cardium_clinical_data_woe_wnm_standarized_f_normalized.json',
 'FetalCLIP_weights.pt',
 'FetalCLIP_config.json',
 'CARDIUM',
 'CARDIUM_embeddings']

In [ ]:
# =============================================================================
# Unzip the CARDIUM IMAGE FILE
# =============================================================================
if not (Path("/content/cardium_images/cardium_images").exists()):
    print("Extracting CARDIUM dataset...")
    !mkdir -p /content/cardium_images
    !tar -xzf "/content/drive/MyDrive/Colab Notebooks/CARDIUM/cardium_images-003.tar.gz" \
        -C /content/cardium_images
else:
    print("Dataset already extracted.")

Extracting CARDIUM dataset...


In [ ]:
!ls /content/cardium_images/cardium_images

fold_1	fold_2	fold_3


In [ ]:
# =============================================================================
# CARDIUM IMAGES
# =============================================================================
# CARDIUM images root:
# The folder from CARDUIM is organized with /cardium_images/fold_1/train/CHD/ or
# /Non_CHD/ and then there is a folder per participant with multiple images within
# the folder
CARDIUM_ROOT = Path("/content/cardium_images/cardium_images")

In [ ]:
# Confirm that the folders exist as expected
print((CARDIUM_ROOT / "fold_1").exists())
print((CARDIUM_ROOT / "fold_1/train/CHD").exists())

True
True


In [ ]:
# Images are stored like this: .../CHD/aedxf00001rrfgkl0/0_aedxf00001rrfgkl0.png
# Because the folder path and file names include identifiers. Pull these
# off for each image.
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def collect_cardium_images(cardium_root: Path):
    # Initialize the list of images - one per row
    rows = []

    # for each fold get the fold number, whether it is training or
    # test data, whether it is a CHD case or not, and the
    # patient ID, image ID and finially image file name.
    for fold_dir in sorted(cardium_root.glob("fold_*")):
        fold = fold_dir.name

        # Train and test split of the data
        for split in ["train", "test"]:
            split_dir = fold_dir / split
            if not split_dir.exists():
                continue

            # Class labels are part of the file path
            for class_name, label in [("CHD", 1), ("Non_CHD", 0)]:
                class_dir = split_dir / class_name
                if not class_dir.exists():
                    continue

                # rglob handles nested patient subfolders
                for img_path in class_dir.rglob("*"):
                    if img_path.suffix.lower() in VALID_EXTS:
                        patient_id = None
                        rel_parts = img_path.relative_to(class_dir).parts
                        if len(rel_parts) >= 2:
                            patient_id = rel_parts[0]  # for example aedxf00001rrfgkl0

                        # Add image to list
                        rows.append({
                            "image_path": str(img_path),
                            "fold": fold,
                            "split": split,
                            "class_name": class_name,
                            "label": label,
                            "patient_id": patient_id,
                            "image_id": img_path.stem,
                            "filename": img_path.name
                        })

    return pd.DataFrame(rows)

In [ ]:
# Create the dataframe of the images with corresponding data for
# path, fold, split, class name, label, patient ID, image ID and filename
df = collect_cardium_images(CARDIUM_ROOT)
print(df.shape)
print(df.head())
print(df["label"].value_counts())

(19674, 8)
                                          image_path    fold  split  \
0  /content/cardium_images/cardium_images/fold_1/...  fold_1  train   
1  /content/cardium_images/cardium_images/fold_1/...  fold_1  train   
2  /content/cardium_images/cardium_images/fold_1/...  fold_1  train   
3  /content/cardium_images/cardium_images/fold_1/...  fold_1  train   
4  /content/cardium_images/cardium_images/fold_1/...  fold_1  train   

  class_name  label         patient_id              image_id  \
0        CHD      1  aedxf00031rrfgkl0  33_aedxf00031rrfgkl0   
1        CHD      1  aedxf00031rrfgkl0  14_aedxf00031rrfgkl0   
2        CHD      1  aedxf00031rrfgkl0   2_aedxf00031rrfgkl0   
3        CHD      1  aedxf00031rrfgkl0  36_aedxf00031rrfgkl0   
4        CHD      1  aedxf00031rrfgkl0  31_aedxf00031rrfgkl0   

                   filename  
0  33_aedxf00031rrfgkl0.png  
1  14_aedxf00031rrfgkl0.png  
2   2_aedxf00031rrfgkl0.png  
3  36_aedxf00031rrfgkl0.png  
4  31_aedxf00031rrfgkl0.png

In [ ]:
# Check counts and percentages by split, and label
# We know from the CARDIUM documentation that the
# images are ~16% from the CHD group. Confirm this is
# correct. (Note: The CHD group makes up 7% of the
# participants, but 16% of the images are from this group)
summary = (
    df.groupby(["split", "label"])
      .size()
      .to_frame("count")
)

summary["percent"] = (
    summary.groupby(level=0)["count"]
           .transform(lambda x: 100 * x / x.sum())
)

display(summary)

count    percent
split label                  
test  0       5490  83.714547
      1       1068  16.285453
train 0      10980  83.714547
      1       2136  16.285453

In [ ]:
# Check counts and percentages by fold, split, and label
summary = (
    df.groupby(["fold", "split", "label"])
      .size()
      .to_frame("count")
)

summary["percent"] = (
    summary.groupby(level=[0,1])["count"]
           .transform(lambda x: 100 * x / x.sum())
)

display(summary)

count    percent
fold   split label                  
fold_1 test  0       1797  83.737185
             1        349  16.262815
       train 0       3693  83.703536
             1        719  16.296464
fold_2 test  0       1964  81.901585
             1        434  18.098415
       train 0       3526  84.759615
             1        634  15.240385
fold_3 test  0       1729  85.849057
             1        285  14.150943
       train 0       3761  82.768486
             1        783  17.231514

In [ ]:
# Set the out directory for the CARDIUM embeddings
OUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings")
OUT_DIR.mkdir(exist_ok=True)

print("OUT_DIR exists:", OUT_DIR.exists())
print("OUT_DIR path:", OUT_DIR)

# Each CARDIUM image will produce a 768-dimensional FetalCLIP embedding.
# Our downstream classifier should expect:
# input dimension = 768
# one row per image
# plus metadata columns for path, fold, split, class name, label, patient ID,
# image ID and filename


OUT_DIR exists: True
OUT_DIR path: /content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings


In [ ]:
# =============================================================================
# FETAL CLIP json and weights file
# =============================================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/CARDIUM")

# Define paths for model configuration and weights
FETALCLIP_CONFIG = DRIVE_ROOT /  "FetalCLIP_config.json"
FETALCLIP_WEIGHTS = DRIVE_ROOT / "FetalCLIP_weights.pt"

In [ ]:
# Set device to GPU (CUDA) for faster computation
device = torch.device("cuda")

# Load and register model configuration
with open(FETALCLIP_CONFIG, "r") as file:
    config_fetalclip = json.load(file)
open_clip.factory._MODEL_CONFIGS["FetalCLIP"] = config_fetalclip

# Load the FetalCLIP model and preprocessing transforms as well as tokenizer
model, preprocess_train, preprocess_test = open_clip.create_model_and_transforms("FetalCLIP", pretrained=FETALCLIP_WEIGHTS)
tokenizer = open_clip.get_tokenizer("FetalCLIP")
# Because we are extracting embedding, use model evaluation mode
# This way we are not using dropout and other training-only behaviors.
model.eval()
model.to(device)

# From the output below, we see that the image encoder is a Vision Transformer
# Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
# is the patch embedding layer. It splits the image into patches and projects
# them into the transformer.
# (transformer): Transformer(
# and (token_embedding): Embedding(49408, 768)
# shows the text side of CLIP is also present,
# but here we are just using the images

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-23): 24 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1024,), eps=1e-05, elementwi

In [ ]:
# Test one image and check size

sample_image = next((CARDIUM_ROOT / "fold_1/train/CHD").rglob("*.png"))

from PIL import Image

img = Image.open(sample_image).convert("RGB")
x = preprocess_test(img).unsqueeze(0).to(device)

with torch.no_grad():
    emb = model.encode_image(x)

print("Embedding shape:", emb.shape)

Embedding shape: torch.Size([1, 768])


In [ ]:
# Create Class for CARDIUM Image dataset
# with metadata as extracted above
# This process takes the rows in fold_df (one row
# per image), opens each image file, applies the
# FetalCLIP image preprocessing, returns image
# tensors plus metadata

class CardiumImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]

        try:
            img = Image.open(img_path).convert("RGB")
            x = self.transform(img)
            ok = 1
        except Exception:
            # fallback in case of a bad file
            x = torch.zeros(3, 224, 224)
            ok = 0

        meta = {
            "image_path": row["image_path"],
            "fold": row["fold"],
            "split": row["split"],
            "class_name": row["class_name"],
            "label": int(row["label"]),
            "patient_id": row["patient_id"],
            "image_id": row["image_id"],
            "filename": row["filename"],
            "ok": ok
        }

        return x, meta

In [ ]:
# Define function to extract embeddings for fold by fold
# by loading the images in batches, passing them through FetalCLIP,
# storing the embedding vector for each image (768 embedding vector),
# and storing the metadata for each image.

# This function takes the fold-specific dataframe, the FetalCLIP model (that
# is already loaded), the inference preprocessing transform, device settings,
# and batch settings. It outputs the following for each fold:
# meta_df -- metadata table
# embeddings -- embedding matrix

def extract_embeddings_for_fold(fold_df, model, preprocess, device, batch_size=64, num_workers=2):
    # Creates dataset object is then used by a PyTorch DataLoader
    # to load images in batches during embedding extraction
    dataset = CardiumImageDataset(fold_df, preprocess)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # Initialize embeddings and metadata
    all_embeddings = []
    all_meta = []

    # As noted above using model evaluation mode
    model.eval()

    # torch.no_grad tells Python that we do not want to compute gradients
    with torch.no_grad():
        # batch_imgs is a tensor containing a batch of preprocessed images.
        for batch_imgs, batch_meta in loader:
            batch_imgs = batch_imgs.to(device)

            # model.encode_image() sends the batch of images through the
            # FetalCLIP image encoder and returns embedding vectors.
            emb = model.encode_image(batch_imgs)

            # Normalize feature vectors
            emb = emb / emb.norm(dim=-1, keepdim=True)

            # The embeddings are a PyTorch tensor on the GPU
            # However, pandas and NumPy storage are easier on CPU
            # parquet export below will use these. So, move the
            # embeddings to CPU
            emb = emb.cpu().numpy()
            batch_size_actual = emb.shape[0]

            # Pair the embedding row with the meta data dictionary
            for i in range(batch_size_actual):
                meta_i = {}
                # batch_meta contains the meta data
                for k in batch_meta:
                    v = batch_meta[k][i]
                    if hasattr(v, "item"):
                        v = v.item()
                    meta_i[k] = v

                all_meta.append(meta_i)
                all_embeddings.append(emb[i])

    # Stack embeddings into a matrix and convert meta data to a dataframe
    embeddings = np.vstack(all_embeddings)
    meta_df = pd.DataFrame(all_meta)

    return meta_df, embeddings

In [ ]:
# Run fold by fold and save output files

for fold_name in sorted(df["fold"].unique()):
    print(f"\nProcessing {fold_name}...")

    fold_df = df[df["fold"] == fold_name].reset_index(drop=True)
    print("Number of images:", len(fold_df))

   # 9 metadata columns plus 768 embedding columns
    meta_df, embeddings = extract_embeddings_for_fold(
        fold_df=fold_df,
        model=model,
        preprocess=preprocess_test,
        device=device,
        batch_size=64,
        num_workers=2
    )

    embed_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embed_df = pd.DataFrame(embeddings, columns=embed_cols)

    full_df = pd.concat([meta_df.reset_index(drop=True), embed_df], axis=1)

    out_file = OUT_DIR / f"cardium_fetalclip_embeddings_{fold_name}.parquet"
    full_df.to_parquet(out_file, index=False)

    print(f"Saved: {out_file}")
    print("Shape:", full_df.shape)


Processing fold_1...
Number of images: 6558
Saved: /content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_1.parquet
Shape: (6558, 777)

Processing fold_2...
Number of images: 6558
Saved: /content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_2.parquet
Shape: (6558, 777)

Processing fold_3...
Number of images: 6558
Saved: /content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_3.parquet
Shape: (6558, 777)


In [ ]:
# Confirm that the files exist on Google Drive so that we can use them
# as frozen embeddings in the next phase of our project

list(OUT_DIR.glob("*.parquet"))

[PosixPath('/content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_1.parquet'),
 PosixPath('/content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_2.parquet'),
 PosixPath('/content/drive/MyDrive/Colab Notebooks/CARDIUM/CARDIUM_embeddings/cardium_fetalclip_embeddings_fold_3.parquet')]

In [ ]:
# Double check the fold 1 datafile that it looks as expected
# 6558 rows = images
# 777 columns = 9 metadata fields + 768 embeddings
fold1_path = OUT_DIR / "cardium_fetalclip_embeddings_fold_1.parquet"

df_fold1 = pd.read_parquet(fold1_path)

df_fold1.shape

(6558, 777)

In [ ]:
# Verify the metadata is included followed by the embeddings
df_fold1.columns[:12]

Index(['image_path', 'fold', 'split', 'class_name', 'label', 'patient_id',
       'image_id', 'filename', 'ok', 'emb_0', 'emb_1', 'emb_2'],
      dtype='object')

In [ ]:
# Confirm we have 768 embedding columns
len([c for c in df_fold1.columns if c.startswith("emb_")])

768

In [ ]:
# Check that the embedding values make sense (i.e. Values should
# be small floating-point numbers, include positive and negative
# values, not all equal zero and not all equal NaN)
df_fold1.filter(like="emb_").iloc[0].head()

,0
emb_0,-0.070888
emb_1,-0.027637
emb_2,-0.063114
emb_3,-0.012052
emb_4,0.015422


In [ ]:
# Confirm that embedding normalization worked as expected
# This should be close to 1
row = df_fold1.filter(like="emb_").iloc[0].values

np.linalg.norm(row)

np.float32(0.99999994)

In [ ]:
# Confirm that nothing funny happened with the class balance

df_fold1["label"].value_counts(normalize=True)

,proportion
label,
0,0.837145
1,0.162855


In [ ]:
# Confirm that patient IDs make sense
# We should have 1103 patients
df_fold1["patient_id"].nunique()

1103

In [ ]:
# Confirm embeddings differ across patients
df_fold1.filter(like="emb_").std().mean()

np.float32(0.024780259)

In [ ]:
# Check cosine similarity between two random images
from sklearn.metrics.pairwise import cosine_similarity

X = df_fold1.filter(like="emb_").iloc[:5]

cosine_similarity(X)

array([[0.9999998 , 0.6067136 , 0.49174044, 0.50059736, 0.5213016 ],
       [0.6067136 , 1.        , 0.39511064, 0.84154797, 0.42391044],
       [0.49174044, 0.39511064, 1.0000002 , 0.3672961 , 0.8928316 ],
       [0.50059736, 0.84154797, 0.3672961 , 0.99999964, 0.46011382],
       [0.5213016 , 0.42391044, 0.8928316 , 0.46011382, 1.0000001 ]],
      dtype=float32)

In [ ]:
# Check simlarity in images for the same patient
embed_cols = [c for c in df_fold1.columns if c.startswith("emb_")]

same_patient = df_fold1[df_fold1["patient_id"] == df_fold1["patient_id"].iloc[0]]
same_patient[["patient_id", "label", "filename"]].head()

,patient_id,label,filename
0,aedxf00031rrfgkl0,1,33_aedxf00031rrfgkl0.png
1,aedxf00031rrfgkl0,1,14_aedxf00031rrfgkl0.png
2,aedxf00031rrfgkl0,1,2_aedxf00031rrfgkl0.png
3,aedxf00031rrfgkl0,1,36_aedxf00031rrfgkl0.png
4,aedxf00031rrfgkl0,1,31_aedxf00031rrfgkl0.png


In [ ]:
embed_cols = [c for c in df_fold1.columns if c.startswith("emb_")]

sample = df_fold1.sample(100, random_state=1)
X = sample[embed_cols].values

sim = cosine_similarity(X)

# off-diagonal values only
off_diag = sim[np.triu_indices_from(sim, k=1)]

print("Min:", off_diag.min())
print("Median:", np.median(off_diag))
print("Max:", off_diag.max())
print("Mean:", off_diag.mean())


# These results are reassurring because even within one patient, fetal
# ultrasound images can differ because they may include different
# views of the heart, different zoom levels, structures in the frame,
# and can be from different gestational ages

Min: -0.124911755
Median: 0.48330486
Max: 0.9325867
Mean: 0.4825609
